### Лабораторна робота №5: Абелеві групи та кільця

**Мета роботи:** Практично вивчити властивості абелевих груп, навчитися працювати з кільцями та визначати їх структуру за допомогою SageMath.

---
### 1. Структура абелевих груп

Розглянемо абелеві групи порядку $400$:
$$
\mathbb{Z}_{400}, \quad \mathbb{Z}_{16}\times\mathbb{Z}_{25}, \quad \mathbb{Z}_{20}\times\mathbb{Z}_{20}, \quad \mathbb{Z}_{4}\times\mathbb{Z}_{100}, \quad \mathbb{Z}_{2}\times\mathbb{Z}_{10}\times\mathbb{Z}_{20}, \quad \mathbb{Z}_{505}^{*}.
$$
Для кожної з цих груп виконайте наступні кроки:
1. Знайдіть елементарні дільники групи.
2. Розкладіть групу у добуток циклічних $p$-груп.
3. Визначте кількість елементів кожного можливого порядку.
4. Порівняйте групи та визначте, які з них є ізоморфними.


In [4]:
groups_data = [
    ("Z400", AdditiveAbelianGroup([400])),
    ("Z16 x Z25", AdditiveAbelianGroup([16, 25])),
    ("Z20 x Z20", AdditiveAbelianGroup([20, 20])),
    ("Z4 x Z100", AdditiveAbelianGroup([4, 100])),
    ("Z2 x Z10 x Z20", AdditiveAbelianGroup([2, 10, 20])),
    ("Z505*", Integers(505).unit_group())
]

isomorphism_classes = {}

for name, G in groups_data:
    print(f"Аналіз групи {name}:")
    # отримуємо інваріантні множники (канонічний вигляд)
    invs = G.invariants()
    
    # 1. Знаходження елементарних дільників (через факторизацію інвентарів)
    elem_divisors = []
    for inv in invs:
        for p, e in factor(inv):
            elem_divisors.append(p**e)
    elem_divisors.sort()
    print(f"1. Елементарні дільники: {elem_divisors}")
    
    # 2. Розклад у добуток циклічних p-груп
    p_group_str = " x ".join([f"Z{d}" for d in elem_divisors])
    print(f"2. Розклад (p-групи): {p_group_str}")
    
    # 3. Підрахунок кількості елементів кожного порядку
    orders_count = {}
    for g in G:
        o = g.order()
        orders_count[o] = orders_count.get(o, 0) + 1
        
    print("3. Кільксть елементів за порядками:")
    orders_str = ", ".join([f"ord {o}: {count}" for o, count in sorted(orders_count.items())])
    print(f"    {orders_str}")
    
    # Зберігаємо інваріанти як унікальний ключ для перевірки ізоморфізму
    iso_key = tuple(invs)
    if iso_key not in isomorphism_classes:
        isomorphism_classes[iso_key] = []
    isomorphism_classes[iso_key].append(name)
    
    print("\n")
    print("---------------------------------------")
    print("\n")
    
        
# 4. Порівняння груп на ізоморфізм
print("\n4. Висновки щодо ізоморфізму")
for key, names in isomorphism_classes.items():
    if len(names) > 1:
        print(f"Групи {', '.join(names)} є ІЗОМОРФНИМИ (мають однакові інваріантні множники: {key}).")
    else:
        print(f"Групи {names[0]} має унікальну структуру серед наведених (інваріантні множники: {key}).")

Аналіз групи Z400:
1. Елементарні дільники: [16, 25]
2. Розклад (p-групи): Z16 x Z25
3. Кільксть елементів за порядками:
    ord 1: 1, ord 2: 1, ord 4: 2, ord 5: 4, ord 8: 4, ord 10: 4, ord 16: 8, ord 20: 8, ord 25: 20, ord 40: 16, ord 50: 20, ord 80: 32, ord 100: 40, ord 200: 80, ord 400: 160


---------------------------------------


Аналіз групи Z16 x Z25:
1. Елементарні дільники: [16, 25]
2. Розклад (p-групи): Z16 x Z25
3. Кільксть елементів за порядками:
    ord 1: 1, ord 2: 1, ord 4: 2, ord 5: 4, ord 8: 4, ord 10: 4, ord 16: 8, ord 20: 8, ord 25: 20, ord 40: 16, ord 50: 20, ord 80: 32, ord 100: 40, ord 200: 80, ord 400: 160


---------------------------------------


Аналіз групи Z20 x Z20:
1. Елементарні дільники: [4, 4, 5, 5]
2. Розклад (p-групи): Z4 x Z4 x Z5 x Z5
3. Кільксть елементів за порядками:
    ord 1: 1, ord 2: 3, ord 4: 12, ord 5: 24, ord 10: 72, ord 20: 288


---------------------------------------


Аналіз групи Z4 x Z100:
1. Елементарні дільники: [4, 4, 25]
2. Ро

---

### 2. Одна абелева група порядку $n$

1. Реалізуйте функцію, яка для заданого натурального числа $n$ обчислює кількість попарно неізоморфних абелевих груп порядку $n$.
2. Знайдіть перші сто таких натуральних чисел $n$, для яких існує рівно одна абелева група порядку $n$. Для кожного знайденого $n$ проаналізуйте його розклад на прості множники.
3. Сформулюйте гіпотезу: для яких $n\in\mathbb{N}$ існує рівно одна (з точністю до ізоморфізму) абелева група порядку $n$.



In [9]:
def count_abelian_groups(n):
    # Отримуємо розклад числа на прості множники (список пар: просте число, степінь)
    factors = factor(n)
    total_groups = 1
    
    for p, k in factors:
        total_groups *= Partitions(k).cardinality()# ця штука рахує кількість розбиттів степення k
    return total_groups

#-------------------------------------

# 1. Реалізація функції та тестування на заданих n
print("1. Реалізація функції та тестування на заданних n:")
test_ns = [10, 16, 20, 64, 100, 144, 150, 300]
for num in test_ns:
    print(f"    n = {num:<3} | Розклад: {str(factor(num)):<12} | Кількість груп: {count_abelian_groups(num)}")
    
# 2. Знаходження перших 100 чисел з рівно 1 групою
print("\n2. Перші 100 чисел n, для яких існує рівно одна абелева група:")
unique_group_ns = []
current_n = 1

while len(unique_group_ns) < 100:
    if count_abelian_groups(current_n) == 1:
        unique_group_ns.append(current_n)
    current_n += 1
    
# Виводимо перші 20 для наочності та їхній розклад
for num in unique_group_ns[:20]:
    print(f"    n = {num:<3} Розклад: {factor(num)}")
print(f"    ... і так далі до n = {unique_group_ns[-1]}")


1. Реалізація функції та тестування на заданних n:
    n = 10  | Розклад: 2 * 5        | Кількість груп: 1
    n = 16  | Розклад: 2^4          | Кількість груп: 5
    n = 20  | Розклад: 2^2 * 5      | Кількість груп: 2
    n = 64  | Розклад: 2^6          | Кількість груп: 11
    n = 100 | Розклад: 2^2 * 5^2    | Кількість груп: 4
    n = 144 | Розклад: 2^4 * 3^2    | Кількість груп: 10
    n = 150 | Розклад: 2 * 3 * 5^2  | Кількість груп: 2
    n = 300 | Розклад: 2^2 * 3 * 5^2 | Кількість груп: 4

2. Перші 100 чисел n, для яких існує рівно одна абелева група:
    n = 1   Розклад: 1
    n = 2   Розклад: 2
    n = 3   Розклад: 3
    n = 5   Розклад: 5
    n = 6   Розклад: 2 * 3
    n = 7   Розклад: 7
    n = 10  Розклад: 2 * 5
    n = 11  Розклад: 11
    n = 13  Розклад: 13
    n = 14  Розклад: 2 * 7
    n = 15  Розклад: 3 * 5
    n = 17  Розклад: 17
    n = 19  Розклад: 19
    n = 21  Розклад: 3 * 7
    n = 22  Розклад: 2 * 11
    n = 23  Розклад: 23
    n = 26  Розклад: 2 * 13
    n = 

3. _Сформулюйте гіпотезу: для яких_ $n\in\mathbb{N}$ _існує рівно одна (з точністю до ізоморфізму) абелева група порядку_ $n$_._

Існує рівно одна абелева група порядку $n$ тоді й лише тоді, коли число $n$ є вільним від квадратів.

Тобто, в його розкладі на прості множники $n={p_1^{k_1}}{p_2^{k_2}}...{p_m^{k_m}}$ всі степені $k_i$ дорівнюють $1$ ($n=p_1•p_2...p_m$, де $p_i$ $-$ різні прості числа).

Обґрунтування: Оскільки загальна кількість неізоморфних абелевих груп порядку $n$ обчислюється як добуток кількостей розбиттів кожного зі степенів $k_i$, єдиний спосіб отримати в результаті $1$ $-$ це мати всі множники рівними $1$. Кількість розбиттів дорівнює одиниці виключно для степення $1$. Наявність будь-якого простого дільника у квадраті (чи у вищому степені) автоматично створює додаткові варіанти розбиття і збільшує загальну кількість груп.


---
### 3. Дільники нуля та одиниці в скінченних кільцях
Розгляньте скінченні кільця $\mathbb{Z}_{30}$, $\mathbb{Z}_{12}\times\mathbb{Z}_{15}$, $M_3(\mathbb{Z}_3)$ та для кожного з них виконайте наступні кроки:
1. Знайдіть дільники нуля та їх кількість.
2. Знайдіть дільники одиниці та їх кількість.
3. Порівняйте отримані результати та зробіть висновок про взаємозв'язок між дільниками нуля та дільниками одиниці в скінченних кільцях.


In [2]:
print("Аналіз кільця Z30:")
R30 = Integers(30)
zero_divs = []
units = []

for x in R30:
    if x == 0:
        continue
    is_unit = False
    is_zd = False
    
    # 1. Шукаємо дільники одиниці
    for y in R30:
        if x * y == 1:
            is_unit = True
            break
            
    # 2. Шукаємо дільники нуля
    for y in R30:
        if y != 0 and x * y == 0:
            is_zd = True
            break
            
    if is_unit:
        units.append(x)
    elif is_zd:
        zero_divs.append(x)
        

# units = [x for x in R30 if x.is_unit()]
# zero_divs = [x for x in R30 if x.is_zero_divisor()]

print(f"1. Дільники нуля ({len(zero_divs)} шт): {zero_divs}")
print(f"2. Дільники одиниці ({len(units)} шт): {units}")

Аналіз кільця Z30:
1. Дільники нуля (21 шт): [2, 3, 4, 5, 6, 8, 9, 10, 12, 14, 15, 16, 18, 20, 21, 22, 24, 25, 26, 27, 28]
2. Дільники одиниці (8 шт): [1, 7, 11, 13, 17, 19, 23, 29]


In [4]:
print("Аналіз кільця Z12xZ15:")
Z12 = Integers(12)
Z15 = Integers(15)

# Генеруємо всі елементи вручну як пари
elements = [(x, y) for x in Z12 for y in Z15]
zero_elem = (Z12(0), Z15(0))
one_elem = (Z12(1), Z15(1))

zero_divs_prod = []
units_prod = []

for x in elements:
    if x == zero_elem:
        continue
        
    is_unit = False
    is_zd = False
    
    # 1. Шукаємо дільники одиниці
    for y in elements:
        prod = (x[0]*y[0], x[1]*y[1])
        if prod == one_elem:
            is_unit = True
            break
            
    # 2. Шукаємо дільники нуля
    for y in elements:
        if y != zero_elem:
            prod = (x[0]*y[0], x[1]*y[1])
            if prod == zero_elem:
                is_zd = True
                break
                
    if is_unit:
        units_prod.append(x)
    elif is_zd:
        zero_divs_prod.append(x)
        

        
print(f"1. Дільники нуля ({len(zero_divs_prod)} шт). Наприклад перші 5: {zero_divs_prod[:5]}")
print(f"2. Дільники одиниці ({len(units_prod)} шт). Наприклад перші 5: {units_prod[:5]}")

Аналіз кільця Z12xZ15:
1. Дільники нуля (147 шт). Наприклад перші 5: [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5)]
2. Дільники одиниці (32 шт). Наприклад перші 5: [(1, 1), (1, 2), (1, 4), (1, 7), (1, 8)]


In [6]:
print("Аналіз кільця матриць M3(Z3):")
M3 = MatrixSpace(Integers(3), 3)

def manual_det_3x3(M):
    # Рахуємо визначник за правилом трикутникі вручну
    d = (M[0,0]*M[1,1]*M[2,2] + M[0,1]*M[1,2]*M[2,0] + M[0,2]*M[1,0]*M[2,1] - M[0,2]*M[1,1]*M[2,0] - M[0,0]*M[1,2]*M[2,1] - M[0,1]*M[1,0]*M[2,2])
    return d


zero_mat = M3.zero()
count_units_mat = 0
count_zd_mat = 0

for M in M3:
    if M == zero_mat:
        continue
        
    det_val = manual_det_3x3(M)
    
    # Якщо визначник не нуль - матриця оборотна (дільник одиниці)
    if det_val != 0:
        count_units_mat += 1
    else:
        # Якщо визначник 0, а сама матриця ненульова - це дільники нуля
        count_zd_mat += 1
        
        
print(f"1. Кількість дільників нуля: {count_zd_mat}")
print(f"2. Кількість дільників одиниці: {count_units_mat}")

Аналіз кільця матриць M3(Z3):


1. Кількість дільників нуля: 8450
2. Кількість дільників одиниці: 11232


3. _Порівняйте отримані результати та зробіть висновок про взаємозв'язок між дільниками нуля та дільниками одиниці в скінченних кільцях._

У будь-якому скінченному кільці кожен ненульовий елемент є або дільником одиниці, або дільником нуля. Тобто для довільного скінченного кільця $R$ виконується рівність $|R|=1+(кількість$ $дільників$ $одиниці)+(кількість$ $дільників$ $нуля)$ (де $1$ – це нульовий елемент).

Обґрунтування: Нехай маємо скінченне кільце $R$ і якийсь його ненульовий елемент $a$. Розглянемо функцію множення на цей елемент: $f(x)=a•x$, де $x$ пробігає всі елементи кільця $R$. Якщо елемент $a$ не є дільником нуля, це означає, що добуток $a•x$ ніколи не дасть $0$ для ненульових $x$. Отже, всі значення $a•x$ будуть різними (функція є ін'єктивною). Оскільки кільце скінченне, якщо функція переводить усі його елементи в різні значення, то вона обов'язково покриє абсолютно всі елементи кільця (є сюр'єктивною).

Це означає, що серед цих значень обов'язково з'явиться одиниця. Тобто знайдеться такий елемент $b$, для якого $a•b=1$. А це і є визначення дільника одиниці. Отже, якщо елемент скінченного кільця не є дільником нуля, він автоматично стає дільником одиниці.


---
### 4. Факторкільця кільця многочленів
Нехай $f(x)=x^4+x+2$. Виконайте наступні кроки для кожного з кілець $R=\mathbb{Q}, \mathbb{Z}_4, \mathbb{Z}_5$:
1. Побудуйте ідеал $I=(f(x))$ у кільці $R[x]$.
2. Побудуйте факторкільце $R/I$ та знайдіть його порядок.
3. Визначте, чи є факторкільце $R/I$ полем.
4. Знайдіть дільники нуля у факторкільці $R/I$.
5. Розкладіь $f(x)$ на лінійні множники над $R$ та порівняйте з дільниками нуля у факторкільці $R/I$.
6. Перевірте, чи елемент $x+I\in R/I$ є оборотним.

In [23]:
print("Аналіз для Q")
R = QQ
P.<x> = R[]
f = x^4 + x + 2

# 1. Будуємо ідеал
I = P.ideal(f)
print(f"1. Ідеал побудовано: I = ({f})")

# 2. Факторкільце
Q.<x_bar> = P.quotient(I)
# перевірка на скінченність
if R.is_finite():
    print(f"2. Порядок факторкільця: {R.cardinality()^f.degree()}")
else:
    print("2. Базове кільце нескінченне, тому порядок факторкільця також нескінченний.")
    
# 3. Чи є полем
is_field_check = Q.is_field()
print(f"3. Чи є Q полем? Відповідь: {is_field_check}.")

# 4. Дільники нуля
# У нескінченному кільці ми не можемо перебрати елементи. Але ми можемо перевірити, чи є кільце цілісним
is_domain = Q.is_integral_domain()
if is_domain:
    print("4. Факторкільце є цілісним")
    print("   За визначенням, у цілісному кільці дільників нуля НЕМАЄ.")
else:
    print("4. Кільце не є цілісним, отже дільники нуля існують.")
    
# 5. Розклад f(x)
factors = f.factor()
print(f"5. Розклад f(x) над Q: {factors}")
if len(list(factors)) == 1:
    print("   Многочлен незвідний, що підтверджує відсутність дільників нуля.")
    
# 6. Оборотність x + I
# Перевіряємо через розширений алгоритм Евкліда
d, u, v = xgcd(x, f)
print(f"6. Перевірка оборотності через НСД(x, f): d = {d}")
if d == 1:
    print(f"   Оскільки НСД = 1, елемент оборотний. Обернений: {u(x_bar)}")
else:
    print("   Елемент не є оборотним.")

Аналіз для Q
1. Ідеал побудовано: I = (x^4 + x + 2)
2. Базове кільце нескінченне, тому порядок факторкільця також нескінченний.
3. Чи є Q полем? Відповідь: True.
4. Факторкільце є цілісним
   За визначенням, у цілісному кільці дільників нуля НЕМАЄ.
5. Розклад f(x) над Q: x^4 + x + 2
   Многочлен незвідний, що підтверджує відсутність дільників нуля.
6. Перевірка оборотності через НСД(x, f): d = 1
   Оскільки НСД = 1, елемент оборотний. Обернений: -1/2*x_bar^3 - 1/2


In [20]:
print("Аналіз для Z4")
R = Integers(4)
P.<x> = R[]
f = x^4 + x + 2

# 1. Будуємо ідеал
I = P.ideal(f)
print(f"1. Ідеал побудовано: I = ({f})")

# 2. Факторкільце
Q.<x_bar> = P.quotient(I)
elements = []
for a in R:
    for b in R:
        for c in R:
            for d in R:
                elements.append(a + b*x_bar + c*x_bar^2 + d*x_bar^3)

print(f"2. Згенерували всі елементи. Кількість: {len(elements)}")
    
# 3. Чи є полем
is_field_check = Q.is_field()
print(f"3. Чи є Q полем? Відповідь: {is_field_check}.")

# 4. Дільники нуля
zero_divisors = set() # Використовуємо set, щоб уникнути дублікатів

for el1 in elements:
    if el1 == 0:
        continue
    for el2 in elements:
        if el2 == 0:
            continue
        if el1 * el2 == 0:
            zero_divisors.add(el1)
            break
            
print(f"4. Знайдено дільників нуля: {len(zero_divisors)}")
if len(zero_divisors) > 0:
    print(f"   Приклад дільників нуля: {list(zero_divisors)[:3]} ...")
    print("   Оскільки дільники нуля існують, кільце НЕ є полем.")
    
# 5. Розклад f(x)
roots = []
for val in R:
    if f(val) == 0: # підставляємо кожне значення з базового кільця
        roots.append(val)
        
print(f"5. Корені f(x) над Z4: {roots}")
    
# 6. Оборотність x + I
# Перевіряємо через розширений алгоритм Евкліда
print("6. Шукаємо обернений до x_bar елемент повним перебором")
inverse_x = None

for el in elements:
    if x_bar * el == 1:
        inverse_x = el
        break

if inverse_x is not None:
    print(f"   Обернений елемент: {inverse_x}")
else:
    print("   x_bar не є оборотним.")

Аналіз для Z4
1. Ідеал побудовано: I = (x^4 + x + 2)
2. Згенерували всі елементи. Кількість: 256
3. Чи є Q полем? Відповідь: False.
4. Знайдено дільників нуля: 207
   Приклад дільників нуля: [2, x_bar^3 + x_bar^2 + x_bar, x_bar^3 + x_bar^2 + x_bar + 1] ...
   Оскільки дільники нуля існують, кільце НЕ є полем.
5. Корені f(x) над Z4: [1, 2]
6. Шукаємо обернений до x_bar елемент повним перебором
   x_bar не є оборотним.


In [22]:
print("Аналіз для Z5")
R = GF(5)
P.<x> = R[]
f = x^4 + x + 2

# 1. Будуємо ідеал
I = P.ideal(f)
print(f"1. Ідеал побудовано: I = ({f})")

# 2. Факторкільце
Q.<x_bar> = P.quotient(I)
elements = []
for a in R:
    for b in R:
        for c in R:
            for d in R:
                elements.append(a + b*x_bar + c*x_bar^2 + d*x_bar^3)

print(f"2. Згенерували всі елементи. Кількість: {len(elements)}")
    
# 3. Чи є полем
is_field_check = Q.is_field()
print(f"3. Чи є Q полем? Відповідь: {is_field_check}.")

# 4. Дільники нуля
zero_divisors = set() # Використовуємо set, щоб уникнути дублікатів

for el1 in elements:
    if el1 == 0:
        continue
    for el2 in elements:
        if el2 == 0:
            continue
        if el1 * el2 == 0:
            zero_divisors.add(el1)
            break
            
print(f"4. Знайдено дільників нуля: {len(zero_divisors)}")
if len(zero_divisors) > 0:
    print(f"   Приклад дільників нуля: {list(zero_divisors)[:3]} ...")
    print("   Оскільки дільники нуля існують, кільце НЕ є полем.")
    
# 5. Розклад f(x)
roots = []
for val in R:
    if f(val) == 0: # підставляємо кожне значення з базового кільця
        roots.append(val)
        
print(f"5. Корені f(x) над Z5: {roots}")
    
# 6. Оборотність x + I
# Перевіряємо через розширений алгоритм Евкліда
print("6. Шукаємо обернений до x_bar елемент повним перебором")
inverse_x = None

for el in elements:
    if x_bar * el == 1:
        inverse_x = el
        break

if inverse_x is not None:
    print(f"   Обернений елемент: {inverse_x}")
else:
    print("   x_bar не є оборотним.")

Аналіз для Z5
1. Ідеал побудовано: I = (x^4 + x + 2)
2. Згенерували всі елементи. Кількість: 625
3. Чи є Q полем? Відповідь: False.


4. Знайдено дільників нуля: 128
   Приклад дільників нуля: [2*x_bar^3 + 2*x_bar^2 + 4*x_bar + 3, 4*x_bar^3 + 2*x_bar^2 + 2*x_bar + 1, 2*x_bar^3 + 4*x_bar^2 + 2*x_bar + 4] ...
   Оскільки дільники нуля існують, кільце НЕ є полем.
5. Корені f(x) над Z5: [2]
6. Шукаємо обернений до x_bar елемент повним перебором
   Обернений елемент: 2*x_bar^3 + 2


---
### 5*. Корінь з $-1$ за модулем $p$

Виконайте наступні кроки і на їх основі сформулюйте гіпотезу: для яких простих чисел $p$ фактор-кільце $\mathbb{Z}_p[x]/(x^2+1)$ є полем, тобто многочлен $x^2+1$ не має коренів у $\mathbb{Z}_p$.
1. Знайдіть перші 100 простих чисел $p$, для яких факторкільце $\mathbb{Z}_p[x]/(x^2+1)$ є полем.
2. Визначте остачі цих простих чисел при діленні на $m=2,3,\ldots,7$ та проаналізуйте отримані результати.


---

### 6*. Розфарбування кубика Рубика $2\times2\times2$

Розглянемо кубик Рубика розміру $2\times2\times2$, який має $24$ зовнішні грані кутиків (наліпки):

![rubik cube](../assets/rubik_cube.jpg)

1. Побудуйте групу поворотів кубика Рубика 2х2х2 як підгрупу симетричної групи $S_{24}$ та знайдіть її порядок.
2. Знайдіть кількість різних розфарбувань кубика у 10 кольорів. Вважайте два розфарбування однаковими, якщо одне можна отримати з іншого за допомогою поворотів кубика.



In [27]:
G = SymmetricGroup(24)
# Базові рухи всіх 6 граней
g1 = G('(1, 2, 4, 3)(5, 24, 9, 7)(6, 23, 10, 8)') # верхня (U)
g2 = G('(15, 17, 18, 16)(19, 13, 11, 22)(14, 12, 21, 20)') # нижня (D)
g3 = G('(10, 9, 11, 12)(3, 23, 18, 13)(7, 4, 21, 17)') # передня (F)
g4 = G('(6, 5, 19, 20)(2, 8, 15, 22)(24, 1, 14, 16)') # задня (B)
g5 = G('(7, 8, 14, 13)(3, 5, 15, 12)(10, 1, 19, 17)') # ліва (L)
g6 = G('(23, 24, 22, 21)(4, 6, 16, 11)(9, 2, 20, 18)') # права (R)

# 1. Будуємо підгрупу і шукаємо порядок
H = G.subgroup([g1, g2, g3, g4, g5, g6]);
order = H.order()
print(f"1. Порядок групи станів кубика 2x2x2: {order}")

# 2. Лема Коші-Форбеніуса-Бернсайда для підрахунку розфарбувань
print("\n2. Рахуємо кількість унікальних розфарбувань")
# використовуємо класи спряжності
colors = 10
total_colorings = 0

for C in H.conjugacy_classes():
    g = C.an_element() # беремо одного представника з класу
    size = C.cardinality() # скільки рухів у цьому класі
    
    tuples = g.cycle_tuples()

    # Рахуємо незалежні цикли
    moved_points = sum(len(c) for c in tuples)
    fixed_points = 24 - moved_points
    cycles_count = len(tuples) + fixed_points

    # Для кожного циклу обираємо 1 з 10 кольорів
    total_colorings += size * (colors ** cycles_count)

unique_colorings = total_colorings // order
print(f"\nКількість унікальних розфарбувань (у 10 кольорів):")
print(f"{unique_colorings}")

1. Порядок групи станів кубика 2x2x2: 88179840

2. Рахуємо кількість унікальних розфарбувань



Кількість унікальних розфарбувань (у 10 кольорів):
12395526079546335
